# FinanceBench RAGAS evaluation with NeMo Retriever Library

This notebook reproduces the FinanceBench evaluation flow from NVIDIA's `evaluation_01_ragas.ipynb`, while using **NeMo Retriever Library (NRL)** to ingest PDFs, retrieve contexts, and generate answers. It reports the same Ragas metrics—Answer Accuracy, Context Relevance, and Response Groundedness—and adds **document-level recall@1, recall@5, and recall@10 before answer generation**.

FinanceBench labels each question with a `doc_name`; therefore recall here means that at least one of the top-*k* NRL hits comes from the labelled source document. This isolates retrieval quality from answer-generation quality.

## 1. Install dependencies and download FinanceBench

Run this from the NeMo Retriever repository root. The `[llm]` extra provides NRL's LiteLLM client. Ragas is retained only to calculate the same three metrics as the reference notebook.

In [ ]:
%pip install -e "./nemo_retriever[llm]" ragas langchain-nvidia-ai-endpoints
!git clone https://github.com/patronus-ai/financebench.git data/financebench

## 2. Configure and ingest with NRL

Set `REBUILD_INDEX=True` only when the LanceDB table does not already contain the FinanceBench PDFs. Ingesting can take several minutes and requires the normal NRL extraction/embedding setup.

In [ ]:
from pathlib import Path
import json
import os

FINANCEBENCH_ROOT = Path("data/financebench")
PDF_DIR = FINANCEBENCH_ROOT / "pdfs"
QA_PATH = FINANCEBENCH_ROOT / "data" / "financebench_open_source.jsonl"
LANCEDB_URI = "lancedb-financebench"
TABLE_NAME = "financebench"
EMBED_MODEL = "nvidia/llama-nemotron-embed-1b-v2"
MAX_QUESTIONS = 50  # Set to None for the complete FinanceBench open-source split.
RETRIEVAL_K = 10    # Must be at least max(1, 5, 10).
REBUILD_INDEX = False

assert PDF_DIR.is_dir(), f"Missing FinanceBench PDFs: {PDF_DIR}"
assert QA_PATH.is_file(), f"Missing FinanceBench labels: {QA_PATH}"

In [ ]:
if REBUILD_INDEX:
    # NRL creates a LanceDB collection containing embedded document chunks.
    !retriever ingest {PDF_DIR} --lancedb-uri {LANCEDB_URI} --table-name {TABLE_NAME} --embed-model-name {EMBED_MODEL}

## 3. Load questions and retrieve with NRL

This cell performs all retrieval first. It preserves the top-10 chunks and metadata for each question; generation never performs another retrieval pass.

In [ ]:
import pandas as pd
from nemo_retriever.graph.retriever import Retriever

with QA_PATH.open() as handle:
    qa_pairs = [json.loads(line) for line in handle]
if MAX_QUESTIONS is not None:
    qa_pairs = qa_pairs[:MAX_QUESTIONS]

retriever = Retriever(
    vdb_kwargs={"uri": LANCEDB_URI, "table_name": TABLE_NAME},
    embed_kwargs={"model_name": EMBED_MODEL, "embed_model_name": EMBED_MODEL},
    top_k=RETRIEVAL_K,
)
print(f"Evaluating {len(qa_pairs)} FinanceBench questions")

In [ ]:
def normalise_document_name(value):
    """Compare FinanceBench doc_name values with NRL source metadata."""
    return Path(str(value or "")).stem.lower()

def source_name(metadata):
    """Return the source identifier populated by NRL/LanceDB."""
    for key in ("source", "source_id", "path", "source_path"):
        if metadata.get(key):
            return normalise_document_name(metadata[key])
    return ""

def document_recall_at_k(hit_metadata, gold_doc_name, k):
    gold = normalise_document_name(gold_doc_name)
    return any(source_name(metadata) == gold for metadata in hit_metadata[:k])

retrieval_rows = []
for index, qa in enumerate(qa_pairs, start=1):
    result = retriever.retrieve(qa["question"], top_k=RETRIEVAL_K)
    retrieval_rows.append({
        "financebench_id": qa["financebench_id"],
        "query": qa["question"],
        "reference": qa["answer"],
        "gold_doc_name": qa["doc_name"],
        "retrieved_contexts": result.chunks,
        "retrieval_metadata": result.metadata,
        "recall_at_1": document_recall_at_k(result.metadata, qa["doc_name"], 1),
        "recall_at_5": document_recall_at_k(result.metadata, qa["doc_name"], 5),
        "recall_at_10": document_recall_at_k(result.metadata, qa["doc_name"], 10),
    })
    if index % 10 == 0 or index == len(qa_pairs):
        print(f"Retrieved {index}/{len(qa_pairs)} questions")

retrieval_df = pd.DataFrame(retrieval_rows)
recall_summary = (retrieval_df[["recall_at_1", "recall_at_5", "recall_at_10"]]
                  .mean().rename(lambda column: column.replace("recall_at_", "Recall@")))
display(recall_summary.to_frame("document_recall"))

## 4. Generate answers with NRL

Set `NVIDIA_API_KEY` (or configure an OpenAI-compatible endpoint through LiteLLM) before running. Generation receives exactly the contexts retrieved above.

In [ ]:
from nemo_retriever.models.llm import LiteLLMClient

assert os.environ.get("NVIDIA_API_KEY"), "Set NVIDIA_API_KEY before generating answers."
generator = LiteLLMClient.from_kwargs(
    model="nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5",
    temperature=0.0,
    max_tokens=512,
)

answers = []
for index, row in retrieval_df.iterrows():
    generated = generator.generate(row.query, row.retrieved_contexts)
    answers.append(generated.answer if generated.error is None else "")
    if (index + 1) % 10 == 0 or index + 1 == len(retrieval_df):
        print(f"Generated {index + 1}/{len(retrieval_df)} answers")

evaluation_records = retrieval_df.assign(response=answers)[
    ["query", "reference", "retrieved_contexts", "response"]
].rename(columns={"query": "user_input"}).to_dict("records")

## 5. Run the same Ragas metrics as the reference notebook

NRL supplied the documents, contexts, and answers. Ragas evaluates the identical metric set from the reference: Answer Accuracy, Context Relevance, and Response Groundedness.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerAccuracy, ContextRelevance, ResponseGroundedness
from ragas.run_config import RunConfig

judge_llm = ChatNVIDIA(model="openai/gpt-oss-120b")
ragas_results = evaluate(
    dataset=EvaluationDataset.from_list(evaluation_records),
    metrics=[AnswerAccuracy(), ContextRelevance(), ResponseGroundedness()],
    llm=LangchainLLMWrapper(judge_llm),
    run_config=RunConfig(max_workers=1, max_wait=120),
)
ragas_results

## 6. Inspect recall alongside answer-quality metrics

Recall is calculated before generation and is not affected by the generator or judge. The per-question table makes retrieval misses visible next to Ragas scores.

In [ ]:
ragas_df = ragas_results.to_pandas()
report_df = pd.concat([
    retrieval_df[["financebench_id", "gold_doc_name", "recall_at_1", "recall_at_5", "recall_at_10"]].reset_index(drop=True),
    ragas_df.reset_index(drop=True),
], axis=1)

display(recall_summary.to_frame("document_recall"))
display(report_df.head())
report_df.to_json("financebench_nrl_ragas_results.jsonl", orient="records", lines=True)